# ccquant — Market Tracker

**Where is the market, and where do regimes suggest it may go?**

This notebook is a fast research surface over DuckDB marts — not a trading
dashboard and not a live price API. It reads `mart_signals_daily` /
`fct_ohlcv_daily` via `ccquant.forecasting` loaders, labels **macro liquidity**
and **BTC on-chain** regimes, and prints a rule-based outlook backed by
*historical* forward returns in similar regime stacks.

**Pipeline (run top-to-bottom):**
1. Setup + load panels (actionable error if signals mart missing)
2. **Now** — snapshot, price context, breadth, OI, regime labels, freshness
3. **Forward** — regime stack, conditional history, outlook text
4. Caveats

Deep forecasts stay in `BTC.ipynb` / `Eth.ipynb` / `Macro.ipynb` / `OnChain_BTC.ipynb`.
Tags print `REAL` / `MISSING` — no silent fake fills for the hero snapshot.

## 0. Setup

In [1]:
from __future__ import annotations

import os
import warnings
from datetime import date, timedelta
from pathlib import Path

import duckdb
import numpy as np
import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv

from ccquant.forecasting import (
    load_daily_panel,
    load_order_book_panel,
    load_signals_panel,
)
from ccquant.mom import full_months, market_mom

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent

load_dotenv(_root / ".env", override=True)

DB_PATH = Path(os.environ.get("CCQUANT_DB", str(_root / "data" / "ccquant.duckdb")))
MOM_LOOKBACK = 12  # weeks
LIQ_LOOKBACK = 52  # weeks (~1y)
FWD_HORIZONS_WK = [4, 13, 26]
BREADTH_LOOKBACK_D = 400
TOP_N = 8
STALE_WARN_DAYS = 3
PLOT_TEMPLATE = "plotly_dark"

print(f"DB:     {DB_PATH}")
print(f"exists: {DB_PATH.is_file()}")

DB:     /home/ricka/Git/GitHub/ccquant/data/ccquant.duckdb
exists: True


In [2]:
def _table_exists(conn: duckdb.DuckDBPyConnection, schema: str, table: str) -> bool:
    row = conn.execute(
        "select count(*) from information_schema.tables"
        " where table_schema = ? and table_name = ?",
        [schema, table],
    ).fetchone()
    return bool(row and row[0] > 0)


def load_macro_series(database: Path) -> pl.DataFrame:
    """Readonly pull of FRED wide mart (long history; date-global)."""
    with duckdb.connect(str(database), read_only=True) as conn:
        if not _table_exists(conn, "main_signals", "fct_macro_series"):
            return pl.DataFrame()
        df = pl.from_arrow(
            conn.execute(
                """
                select date, m2sl, walcl, dgs10, dgs2, t10yie, fedfunds, dtwexbgs, vixcls
                from main_signals.fct_macro_series
                order by date
                """
            ).to_arrow_table()
        )
    return df if isinstance(df, pl.DataFrame) else df.to_frame()


def load_onchain_series(database: Path) -> pl.DataFrame:
    """Readonly BTC on-chain signals (may be short / empty)."""
    with duckdb.connect(str(database), read_only=True) as conn:
        if not _table_exists(conn, "main_signals", "fct_onchain_signals"):
            return pl.DataFrame()
        df = pl.from_arrow(
            conn.execute(
                """
                select date, hashrate, mvrv, nupl, active_addresses, fees_usd,
                       miner_revenue_usd, realized_price
                from main_signals.fct_onchain_signals
                order by date
                """
            ).to_arrow_table()
        )
    return df if isinstance(df, pl.DataFrame) else df.to_frame()


def load_ops_freshness(database: Path) -> pl.DataFrame:
    with duckdb.connect(str(database), read_only=True) as conn:
        if not _table_exists(conn, "main_marts", "mart_ops_freshness"):
            return pl.DataFrame()
        df = pl.from_arrow(
            conn.execute(
                "select * from main_marts.mart_ops_freshness order by domain, entity_key"
            ).to_arrow_table()
        )
    return df if isinstance(df, pl.DataFrame) else df.to_frame()


def z_expr(col: str) -> pl.Expr:
    return (pl.col(col) - pl.col(col).mean()) / (pl.col(col).std() + 1e-12)


def pct_ret(series: pl.Series, lag: int) -> float | None:
    if series.len() <= lag:
        return None
    a, b = float(series[-(lag + 1)]), float(series[-1])
    if a == 0 or np.isnan(a) or np.isnan(b):
        return None
    return b / a - 1.0


def fmt_pct(x: float | None) -> str:
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{100 * x:+.2f}%"


def empty_note(section: str, reason: str) -> None:
    print(f"[MISSING] {section}: {reason}")

In [3]:
if not DB_PATH.is_file():
    raise FileNotFoundError(
        f"DuckDB not found at {DB_PATH}. Run: uv run ccquant sync all"
    )

try:
    signals = load_signals_panel(DB_PATH)
except Exception as exc:
    raise RuntimeError(
        "Failed to load main_marts.mart_signals_daily. "
        "Run: uv run dbt build --project-dir dbt --profiles-dir dbt "
        "(or uv run ccquant sync all without --no-dbt)."
    ) from exc

if signals.is_empty():
    raise RuntimeError(
        "mart_signals_daily is empty. Sync OHLCV + run dbt build, then re-open."
    )

daily = load_daily_panel(DB_PATH)
macro_raw = load_macro_series(DB_PATH)
onchain_raw = load_onchain_series(DB_PATH)
ops = load_ops_freshness(DB_PATH)
try:
    depth = load_order_book_panel(DB_PATH)
except Exception:
    depth = pl.DataFrame()

signals = signals.with_columns(pl.col("date").cast(pl.Date))
daily = daily.with_columns(pl.col("date").cast(pl.Date))

TAG_SIGNALS = "REAL"
TAG_MACRO = "REAL" if not macro_raw.is_empty() else "MISSING"
TAG_ONCHAIN = "REAL" if not onchain_raw.is_empty() else "MISSING"
TAG_OPS = "REAL" if not ops.is_empty() else "MISSING"
TAG_DEPTH = "REAL" if not depth.is_empty() else "MISSING"

print(f"signals:  {signals.shape}  {signals['date'].min()} -> {signals['date'].max()}  [{TAG_SIGNALS}]")
print(f"daily:    {daily.shape}  {daily['date'].min()} -> {daily['date'].max()}  [REAL]")
print(f"macro:    {macro_raw.shape}  [{TAG_MACRO}]")
print(f"onchain:  {onchain_raw.shape}  [{TAG_ONCHAIN}]")
print(f"ops:      {ops.shape}  [{TAG_OPS}]")
print(f"depth:    {depth.shape}  [{TAG_DEPTH}]")
print(f"symbols:  {signals['symbol'].n_unique()}")

signals:  (617, 44)  2026-06-12 -> 2026-07-18  [REAL]
daily:    (74212, 8)  2015-07-20 -> 2026-07-18  [REAL]
macro:    (16647, 9)  [REAL]
onchain:  (30, 8)  [REAL]
ops:      (224, 5)  [REAL]
depth:    (32, 14)  [REAL]
symbols:  49


## Act 1 — Where is the market?

Hero: one headline (risk-on / mixed / risk-off), one supporting sentence, then
snapshot metrics. On-chain and macro labels are **date-global** (BTC network /
FRED), not per-alt fundamentals.

### 1. Now — market snapshot

In [4]:
btc_d = (
    daily.filter(pl.col("symbol") == "BTC")
    .sort("date")
    .unique(subset=["date"], keep="last")
)
eth_d = (
    daily.filter(pl.col("symbol") == "ETH")
    .sort("date")
    .unique(subset=["date"], keep="last")
)

as_of = btc_d["date"][-1]
btc_close = float(btc_d["close"][-1])
closes = btc_d["close"]

ret_1d = pct_ret(closes, 1)
ret_7d = pct_ret(closes, 7)
ret_30d = pct_ret(closes, 30)
ytd_start = date(as_of.year, 1, 1)
btc_ytd = btc_d.filter(pl.col("date") >= ytd_start)
ret_ytd = None
if btc_ytd.height >= 2:
    ret_ytd = float(btc_ytd["close"][-1]) / float(btc_ytd["close"][0]) - 1.0

# Universe breadth from latest overlapping dates in daily panel
latest_dates = daily.select(pl.col("date").max()).item()
look_7 = latest_dates - timedelta(days=7)
look_30 = latest_dates - timedelta(days=30)

# Per-symbol last close vs close ~7d / ~30d ago
sym_last = (
    daily.sort("date")
    .group_by("symbol")
    .agg(
        pl.col("date").last().alias("last_date"),
        pl.col("close").last().alias("last_close"),
        pl.col("volume").last().alias("last_volume"),
    )
)

def _ret_vs_lag(lag_days: int) -> pl.DataFrame:
    target = latest_dates - timedelta(days=lag_days)
    lagged = (
        daily.filter(pl.col("date") <= target)
        .sort("date")
        .group_by("symbol")
        .agg(pl.col("close").last().alias("lag_close"))
    )
    return (
        sym_last.join(lagged, on="symbol", how="inner")
        .with_columns(
            (pl.col("last_close") / pl.col("lag_close") - 1.0).alias("ret")
        )
        .filter(pl.col("lag_close") > 0)
    )

r7 = _ret_vs_lag(7)
r30 = _ret_vs_lag(30)
pct_up_7 = float((r7["ret"] > 0).mean()) if r7.height else None
pct_up_30 = float((r30["ret"] > 0).mean()) if r30.height else None

# Headline from short-term tape + breadth (refined after regimes later)
risk_bits = []
if ret_7d is not None:
    risk_bits.append(1 if ret_7d > 0.02 else (-1 if ret_7d < -0.02 else 0))
if pct_up_7 is not None:
    risk_bits.append(1 if pct_up_7 > 0.55 else (-1 if pct_up_7 < 0.45 else 0))
score_now = sum(risk_bits) if risk_bits else 0
if score_now >= 2:
    HEADLINE = "Risk-on"
elif score_now <= -2:
    HEADLINE = "Risk-off"
else:
    HEADLINE = "Mixed"

print("ccquant · Market Tracker")
print("=" * 48)
print(f"As of {as_of}  |  {HEADLINE}")
print(
    f"BTC ${btc_close:,.2f}  ·  1d {fmt_pct(ret_1d)}  ·  7d {fmt_pct(ret_7d)}"
    f"  ·  30d {fmt_pct(ret_30d)}  ·  YTD {fmt_pct(ret_ytd)}"
)
print(
    f"Universe up 7d: {fmt_pct(pct_up_7)} of {r7.height}  |  "
    f"up 30d: {fmt_pct(pct_up_30)} of {r30.height}"
)
print(
    "Tape is broad." if (pct_up_7 or 0) > 0.55
    else ("Tape is narrow." if (pct_up_7 or 1) < 0.45 else "Breadth is balanced.")
)

gainers = r7.sort("ret", descending=True).head(TOP_N).select("symbol", "ret", "last_close")
losers = r7.sort("ret", descending=False).head(TOP_N).select("symbol", "ret", "last_close")
print("\nTop 7d gainers:")
print(gainers.with_columns((pl.col("ret") * 100).round(2).alias("ret_pct")).drop("ret"))
print("\nTop 7d losers:")
print(losers.with_columns((pl.col("ret") * 100).round(2).alias("ret_pct")).drop("ret"))

ccquant · Market Tracker
As of 2026-07-18  |  Mixed
BTC $64,452.09  ·  1d +0.87%  ·  7d +28.90%  ·  30d +28.90%  ·  YTD -27.37%
Universe up 7d: +36.00% of 50  |  up 30d: +36.00% of 50
Tape is narrow.

Top 7d gainers:
shape: (8, 3)
┌────────┬────────────┬─────────┐
│ symbol ┆ last_close ┆ ret_pct │
│ ---    ┆ ---        ┆ ---     │
│ str    ┆ f64        ┆ f64     │
╞════════╪════════════╪═════════╡
│ BTC    ┆ 64452.09   ┆ 28.9    │
│ PUMP   ┆ 0.001652   ┆ 18.85   │
│ ZEC    ┆ 557.68     ┆ 9.68    │
│ VVV    ┆ 11.4219    ┆ 9.07    │
│ ONDO   ┆ 0.34931    ┆ 7.8     │
│ CRO    ┆ 0.05916    ┆ 6.1     │
│ SKY    ┆ 0.0613     ┆ 5.91    │
│ LTC    ┆ 47.12      ┆ 5.63    │
└────────┴────────────┴─────────┘

Top 7d losers:
shape: (8, 3)
┌────────┬────────────┬─────────┐
│ symbol ┆ last_close ┆ ret_pct │
│ ---    ┆ ---        ┆ ---     │
│ str    ┆ f64        ┆ f64     │
╞════════╪════════════╪═════════╡
│ ETH    ┆ 1855.28    ┆ -38.16  │
│ SOL    ┆ 75.17      ┆ -24.83  │
│ HYPE   ┆ 59.81      ┆ -

### 2. Price context — BTC and ETH

In [5]:
# Event dates from signals panel (date-global flags)
event_dates: list[date] = []
if {"has_positive_event", "has_negative_event", "date"}.issubset(set(signals.columns)):
    ev = (
        signals.filter(
            (pl.col("has_positive_event").fill_null(0) > 0)
            | (pl.col("has_negative_event").fill_null(0) > 0)
            | (pl.col("event_count").fill_null(0) > 0)
        )
        .select("date")
        .unique()
        .sort("date")
    )
    event_dates = ev["date"].to_list()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=btc_d["date"], y=btc_d["close"], name="BTC",
        line=dict(color="#F7931A", width=2),
    )
)
if eth_d.height:
    fig.add_trace(
        go.Scatter(
            x=eth_d["date"], y=eth_d["close"], name="ETH",
            line=dict(color="#627EEA", width=1.5), yaxis="y2",
        )
    )
for d in event_dates[-12:]:  # keep chart readable
    fig.add_vline(x=d, line_width=1, line_dash="dot", line_color="#888", opacity=0.5)

fig.update_layout(
    title="BTC (and ETH) close — event markers when present",
    template=PLOT_TEMPLATE,
    height=420,
    yaxis=dict(title="BTC USD", type="log"),
    yaxis2=dict(title="ETH USD", overlaying="y", side="right", type="log", showgrid=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    margin=dict(l=60, r=60, t=60, b=40),
)
fig.show()
print(f"Event marker dates used: {len(event_dates[-12:])} (of {len(event_dates)} flagged)  [signals]")

Event marker dates used: 0 (of 0 flagged)  [signals]


### 3. Breadth / momentum — is the tape broad or narrow?

In [6]:
# Daily % of symbols above 50d / 200d MA
panel = (
    daily.sort(["symbol", "date"])
    .unique(subset=["symbol", "date"], keep="last")
    .with_columns(
        pl.col("close").rolling_mean(50).over("symbol").alias("ma50"),
        pl.col("close").rolling_mean(200).over("symbol").alias("ma200"),
    )
    .filter(pl.col("ma50").is_not_null())
)

breadth_daily = (
    panel.group_by("date")
    .agg(
        (pl.col("close") > pl.col("ma50")).mean().alias("pct_above_50"),
        (
            pl.when(pl.col("ma200").is_not_null())
            .then(pl.col("close") > pl.col("ma200"))
            .otherwise(None)
        )
        .drop_nulls()
        .mean()
        .alias("pct_above_200"),
        pl.len().alias("n"),
    )
    .sort("date")
)

# Focus recent window for chart readability while keeping full series for stack
breadth_plot = breadth_daily.tail(BREADTH_LOOKBACK_D)
latest_b = breadth_daily.tail(1)
pct50 = float(latest_b["pct_above_50"][0]) if latest_b.height else None
pct200 = (
    float(latest_b["pct_above_200"][0])
    if latest_b.height and latest_b["pct_above_200"][0] is not None
    else None
)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=breadth_plot["date"], y=breadth_plot["pct_above_50"] * 100,
        name="% above 50d MA", line=dict(color="#4CAF50", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=breadth_plot["date"], y=breadth_plot["pct_above_200"] * 100,
        name="% above 200d MA", line=dict(color="#2196F3", width=1.5),
    )
)
fig.add_hline(y=50, line_dash="dash", line_color="#666")
fig.update_layout(
    title="Cross-sectional breadth — share of universe above moving averages",
    template=PLOT_TEMPLATE,
    height=380,
    yaxis=dict(title="% of symbols", range=[0, 100]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    margin=dict(l=60, r=40, t=60, b=40),
)
fig.show()
print(f"Latest breadth: {fmt_pct(pct50)} above 50d  |  {fmt_pct(pct200)} above 200d  [REAL]")

mom = market_mom(daily, as_of=as_of)
mom_full = full_months(mom)
if mom_full.height:
    last_m = mom_full.tail(1)
    n_up = int(last_m["n_up"][0])
    n_sym = int(last_m["n_symbols"][0])
    eq = last_m["eq_weight_mom"][0]
    print(
        f"Last full month MoM breadth: {n_up}/{n_sym} up  ·  "
        f"eq-weight MoM {fmt_pct(float(eq) if eq is not None else None)}"
    )
else:
    empty_note("monthly MoM breadth", "insufficient full months")

breadth_bullish = 1 if (pct50 or 0) > 0.55 else (-1 if (pct50 or 1) < 0.45 else 0)

Latest breadth: +14.89% above 50d  |  +15.62% above 200d  [REAL]
Last full month MoM breadth: 6/49 up  ·  eq-weight MoM -17.70%


### 4. Positioning — open interest

In [7]:
oi_cols = {"symbol", "date", "total_open_interest_usd"}
if not oi_cols.issubset(set(signals.columns)):
    empty_note("open interest", "columns missing on signals panel")
    oi_btc = pl.DataFrame()
    oi_eth = pl.DataFrame()
else:
    oi = signals.select(["symbol", "date", "total_open_interest_usd"]).drop_nulls(
        subset=["total_open_interest_usd"]
    )
    oi_btc = oi.filter(pl.col("symbol") == "BTC").sort("date")
    oi_eth = oi.filter(pl.col("symbol") == "ETH").sort("date")

if oi_btc.is_empty() and oi_eth.is_empty():
    empty_note("open interest", "no non-null OI rows (sync oi + dbt build)")
else:
    fig = go.Figure()
    if not oi_btc.is_empty():
        fig.add_trace(
            go.Scatter(
                x=oi_btc["date"], y=oi_btc["total_open_interest_usd"],
                name="BTC OI USD", line=dict(color="#F7931A"),
            )
        )
    if not oi_eth.is_empty():
        fig.add_trace(
            go.Scatter(
                x=oi_eth["date"], y=oi_eth["total_open_interest_usd"],
                name="ETH OI USD", line=dict(color="#627EEA"),
            )
        )
    fig.update_layout(
        title="Open interest (USD) — BTC / ETH from signals panel",
        template=PLOT_TEMPLATE,
        height=360,
        yaxis=dict(title="OI USD"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
        margin=dict(l=60, r=40, t=60, b=40),
    )
    fig.show()
    if not oi_btc.is_empty():
        print(f"BTC OI latest: ${float(oi_btc['total_open_interest_usd'][-1]):,.0f}  [REAL]")
    if not oi_eth.is_empty():
        print(f"ETH OI latest: ${float(oi_eth['total_open_interest_usd'][-1]):,.0f}  [REAL]")

BTC OI latest: $1,000,000  [REAL]
ETH OI latest: $1,000,000  [REAL]


### 5. Regime labels (now)

Macro liquidity regime ports the Macro notebook composite (expanding when
12-week momentum of the liquidity index is positive). On-chain cycle uses a
**simplified** composite from mart columns (`mvrv`, `nupl`, activity) — not the
full Glassnode stack in `OnChain_BTC.ipynb`.

In [8]:
# --- Macro liquidity regime (weekly) ---
liq_regime: int | None = None
liq_index_last: float | None = None
liq_mom_last: float | None = None
macro_weekly = pl.DataFrame()

if macro_raw.is_empty():
    empty_note("macro regime", "fct_macro_series missing — run sync macro + dbt")
else:
    m = (
        macro_raw.with_columns(pl.col("date").cast(pl.Date))
        .sort("date")
        .with_columns(pl.col("date").dt.truncate("1w").alias("week"))
        .group_by("week")
        .agg(
            [
                pl.col(c).drop_nulls().last().alias(c)
                for c in [
                    "m2sl", "walcl", "dgs10", "dgs2", "t10yie",
                    "fedfunds", "dtwexbgs", "vixcls",
                ]
                if c in macro_raw.columns
            ]
        )
        .sort("week")
        .rename({"week": "date"})
    )
    need = {"m2sl", "walcl", "dgs10", "t10yie"}
    if need.issubset(set(m.columns)):
        m = m.with_columns(
            (pl.col("dgs10") - pl.col("t10yie")).alias("real_10y"),
        ).with_columns(
            (pl.col("m2sl").log() - pl.col("m2sl").log().shift(LIQ_LOOKBACK)).alias(
                "m2_grow_yoy"
            ),
            (pl.col("walcl").log() - pl.col("walcl").log().shift(LIQ_LOOKBACK)).alias(
                "fedbs_grow_yoy"
            ),
            (pl.col("real_10y") - pl.col("real_10y").shift(LIQ_LOOKBACK)).alias(
                "real_rate_delta"
            ),
        ).drop_nulls(subset=["m2_grow_yoy", "fedbs_grow_yoy", "real_rate_delta"])

        if m.height > MOM_LOOKBACK + 5:
            m = m.with_columns(
                (
                    z_expr("m2_grow_yoy")
                    + z_expr("fedbs_grow_yoy")
                    - z_expr("real_rate_delta")
                ).alias("liq_raw")
            )
            mu, sd = float(m["liq_raw"].mean()), float(m["liq_raw"].std())
            m = m.with_columns(
                ((pl.col("liq_raw") - mu) / (sd if sd > 1e-12 else 1.0)).alias(
                    "liq_index"
                ),
            ).with_columns(
                (pl.col("liq_index") - pl.col("liq_index").shift(MOM_LOOKBACK)).alias(
                    "liq_mom"
                ),
            ).with_columns(
                ((pl.col("liq_mom") > 0).cast(pl.Int8)).alias("liq_regime"),
            )
            macro_weekly = m
            liq_regime = int(m["liq_regime"][-1])
            liq_index_last = float(m["liq_index"][-1])
            liq_mom_last = float(m["liq_mom"][-1])
            print(
                f"Macro liquidity: {'EXPANDING' if liq_regime == 1 else 'CONTRACTING'}"
                f"  ·  liq_index={liq_index_last:.2f}  ·  liq_mom={liq_mom_last:.2f}"
                f"  [REAL · date-global FRED]"
            )
        else:
            empty_note("macro regime", "insufficient history after YoY lags")
    else:
        empty_note("macro regime", f"missing columns {need - set(m.columns)}")

# --- On-chain cycle regime (simplified; BTC network metrics) ---
oc_regime: int | None = None
cycle_index_last: float | None = None
cycle_mom_last: float | None = None
oc_weekly = pl.DataFrame()

oc_src = onchain_raw
if oc_src.is_empty() and {"mvrv", "nupl"}.issubset(set(signals.columns)):
    oc_src = (
        signals.filter(pl.col("symbol") == "BTC")
        .select(["date", "mvrv", "nupl", "active_addresses", "hashrate", "fees_usd"])
        .unique(subset=["date"], keep="last")
    )

if oc_src.is_empty():
    empty_note("on-chain regime", "no on-chain rows — migrate onchain / sync + dbt")
else:
    oc = oc_src.with_columns(pl.col("date").cast(pl.Date)).sort("date")
    span_days = (oc["date"][-1] - oc["date"][0]).days if oc.height > 1 else 0
    if span_days >= 90:
        agg_cols = [c for c in oc.columns if c != "date"]
        oc = (
            oc.with_columns(pl.col("date").dt.truncate("1w").alias("week"))
            .group_by("week")
            .agg([pl.col(c).drop_nulls().last().alias(c) for c in agg_cols])
            .sort("week")
            .rename({"week": "date"})
        )
    candidates = [
        c for c in ("mvrv", "nupl", "active_addresses", "hashrate", "fees_usd")
        if c in oc.columns
    ]
    feat_cols = [
        c for c in candidates if oc[c].drop_nulls().len() >= 5
    ]
    if not feat_cols:
        empty_note("on-chain regime", "no usable non-null on-chain features")
    else:
        oc = oc.drop_nulls(subset=feat_cols)
        varying = [c for c in feat_cols if float(oc[c].std() or 0.0) > 1e-12]
        if not varying:
            empty_note(
                "on-chain regime",
                "mvrv/nupl present but constant (no variance) — refresh on-chain extract",
            )
        else:
            mom_lb = min(MOM_LOOKBACK, max(3, oc.height // 3))
            expr = z_expr(varying[0])
            for c in varying[1:]:
                expr = expr + z_expr(c)
            oc = oc.with_columns(expr.alias("cycle_raw"))
            mu, sd = float(oc["cycle_raw"].mean()), float(oc["cycle_raw"].std() or 0.0)
            oc = oc.with_columns(
                ((pl.col("cycle_raw") - mu) / (sd if sd > 1e-12 else 1.0)).alias(
                    "cycle_index"
                ),
            ).with_columns(
                (pl.col("cycle_index") - pl.col("cycle_index").shift(mom_lb)).alias(
                    "cycle_mom"
                ),
            ).with_columns(
                ((pl.col("cycle_mom") > 0).cast(pl.Int8)).alias("oc_regime"),
            )
            oc_weekly = oc
            last_mom = oc["cycle_mom"][-1]
            if last_mom is not None and not (isinstance(last_mom, float) and np.isnan(last_mom)):
                oc_regime = int(oc["oc_regime"][-1])
                cycle_index_last = float(oc["cycle_index"][-1])
                cycle_mom_last = float(last_mom)
                print(
                    f"On-chain cycle: {'BULLISH mom' if oc_regime == 1 else 'BEARISH mom'}"
                    f"  ·  cycle_index={cycle_index_last:.2f}  ·  cycle_mom={cycle_mom_last:.2f}"
                    f"  [REAL · BTC network by date — not per-alt]"
                )
                if span_days < 90:
                    print(
                        f"  note: on-chain span only {span_days}d — treat regime as tentative"
                    )
            else:
                empty_note("on-chain regime", "cycle momentum not yet available (short series)")

print("\nRegime card (now):")
print(
    "  liquidity : "
    + (
        f"{'expanding' if liq_regime == 1 else 'contracting'}"
        if liq_regime is not None
        else "MISSING"
    )
)
print(
    "  on-chain  : "
    + (
        f"{'bullish mom' if oc_regime == 1 else 'bearish mom'}"
        if oc_regime is not None
        else "MISSING"
    )
)
print(
    "  breadth   : "
    + (
        "broad"
        if breadth_bullish == 1
        else ("narrow" if breadth_bullish == -1 else "balanced")
    )
)

Macro liquidity: CONTRACTING  ·  liq_index=-1.61  ·  liq_mom=-1.13  [REAL · date-global FRED]
[MISSING] on-chain regime: mvrv/nupl present but constant (no variance) — refresh on-chain extract

Regime card (now):
  liquidity : contracting
  on-chain  : MISSING
  breadth   : narrow


### 6. Data freshness

In [9]:
today = date.today()
if not ops.is_empty() and "latest_at" in ops.columns:
    # Show a compact domain summary
    summary = (
        ops.with_columns(pl.col("latest_at").cast(pl.Utf8).str.slice(0, 10).alias("asof"))
        .group_by("domain")
        .agg(pl.col("asof").max().alias("latest"), pl.len().alias("entities"))
        .sort("domain")
    )
    print("Ops freshness by domain [REAL]:")
    print(summary)
    # Staleness vs BTC daily
    btc_ops = ops.filter(pl.col("entity_key").cast(pl.Utf8).str.contains("BTC:1d"))
    if btc_ops.height:
        latest_s = str(btc_ops["latest_at"][0])[:10]
        try:
            latest_d = date.fromisoformat(latest_s)
            age = (today - latest_d).days
            if age > STALE_WARN_DAYS:
                print(f"WARNING: BTC daily last refresh age {age}d > {STALE_WARN_DAYS}d")
            else:
                print(f"BTC daily freshness OK (age {age}d)")
        except ValueError:
            print(f"BTC daily latest_at={latest_s}")
else:
    empty_note("ops freshness", "mart_ops_freshness missing — falling back to panel max dates")
    print(f"  daily max date : {daily['date'].max()}")
    print(f"  signals max    : {signals['date'].max()}")
    age = (today - as_of).days
    if age > STALE_WARN_DAYS:
        print(f"WARNING: BTC panel age {age}d — run uv run ccquant sync all")

Ops freshness by domain [REAL]:
shape: (4, 3)
┌─────────┬────────────┬──────────┐
│ domain  ┆ latest     ┆ entities │
│ ---     ┆ ---        ┆ ---      │
│ str     ┆ str        ┆ u32      │
╞═════════╪════════════╪══════════╡
│ macro   ┆ 2026-07-11 ┆ 1        │
│ ohlcv   ┆ 2026-07-18 ┆ 113      │
│ onchain ┆ 2026-07-11 ┆ 1        │
│ wallet  ┆ 2026-07-18 ┆ 109      │
└─────────┴────────────┴──────────┘
BTC daily freshness OK (age 1d)


## Act 2 — Where might it go?

Honest framing: **conditional historical** BTC forward returns when a similar
regime stack prevailed — not a guarantee of the next move. Deep probabilistic
fans live in `BTC.ipynb`.

### 7. Regime stack

In [10]:
# Components: +1 expanding/bullish/broad, -1 contracting/bearish/narrow, 0 unknown/flat
liq_s = 0 if liq_regime is None else (1 if liq_regime == 1 else -1)
oc_s = 0 if oc_regime is None else (1 if oc_regime == 1 else -1)
br_s = breadth_bullish
stack_score = liq_s + oc_s + br_s  # -3 .. +3

if stack_score >= 2:
    STACK_LABEL = "Constructive"
elif stack_score <= -2:
    STACK_LABEL = "Defensive"
else:
    STACK_LABEL = "Neutral / mixed"

fig = go.Figure(
    data=[
        go.Bar(
            x=["Liquidity", "On-chain", "Breadth", "Stack"],
            y=[liq_s, oc_s, br_s, stack_score],
            marker_color=["#42A5F5", "#AB47BC", "#66BB6A", "#FFA726"],
        )
    ]
)
fig.update_layout(
    title=f"Regime stack — {STACK_LABEL} (score {stack_score:+d})",
    template=PLOT_TEMPLATE,
    height=340,
    yaxis=dict(title="signal (−1 / 0 / +1; stack −3..+3)", range=[-3.5, 3.5]),
    margin=dict(l=60, r=40, t=60, b=40),
)
fig.show()
print(f"Stack: {STACK_LABEL}  (liq={liq_s:+d}, onchain={oc_s:+d}, breadth={br_s:+d})")

Stack: Defensive  (liq=-1, onchain=+0, breadth=-1)


### 8. Conditional history — BTC forward returns in similar stacks

In [11]:
# Build weekly BTC + historical stack scores (macro + on-chain + breadth proxy)
btc_w = (
    btc_d.with_columns(pl.col("date").dt.truncate("1w").alias("week"))
    .group_by("week")
    .agg(pl.col("close").last().alias("close"))
    .sort("week")
    .rename({"week": "date"})
    .with_columns(pl.col("close").log().alias("log_close"))
)

# Weekly breadth: mean pct_above_50 in week
br_w = (
    breadth_daily.with_columns(pl.col("date").dt.truncate("1w").alias("week"))
    .group_by("week")
    .agg(pl.col("pct_above_50").mean().alias("pct_above_50"))
    .sort("week")
    .rename({"week": "date"})
)

hist = btc_w.join(br_w, on="date", how="left")
if not macro_weekly.is_empty():
    hist = hist.join(
        macro_weekly.select(["date", "liq_regime"]), on="date", how="left"
    )
else:
    hist = hist.with_columns(pl.lit(None).cast(pl.Int8).alias("liq_regime"))

if not oc_weekly.is_empty() and "oc_regime" in oc_weekly.columns:
    hist = hist.join(
        oc_weekly.select(["date", "oc_regime"]), on="date", how="left"
    )
else:
    hist = hist.with_columns(pl.lit(None).cast(pl.Int8).alias("oc_regime"))

hist = hist.with_columns(
    pl.when(pl.col("pct_above_50") > 0.55)
    .then(pl.lit(1))
    .when(pl.col("pct_above_50") < 0.45)
    .then(pl.lit(-1))
    .otherwise(pl.lit(0))
    .cast(pl.Int8)
    .alias("br_sig"),
    pl.when(pl.col("liq_regime").is_null())
    .then(pl.lit(0))
    .when(pl.col("liq_regime") == 1)
    .then(pl.lit(1))
    .otherwise(pl.lit(-1))
    .cast(pl.Int8)
    .alias("liq_sig"),
    pl.when(pl.col("oc_regime").is_null())
    .then(pl.lit(0))
    .when(pl.col("oc_regime") == 1)
    .then(pl.lit(1))
    .otherwise(pl.lit(-1))
    .cast(pl.Int8)
    .alias("oc_sig"),
).with_columns(
    (pl.col("liq_sig") + pl.col("oc_sig") + pl.col("br_sig")).alias("stack")
)

# Forward log returns
for h in FWD_HORIZONS_WK:
    hist = hist.with_columns(
        (pl.col("log_close").shift(-h) - pl.col("log_close")).alias(f"fwd_{h}w")
    )

# Match "same stack" — prefer exact score; if n small, match sign bucket
def _bucket(s: int) -> str:
    if s >= 2:
        return "constructive"
    if s <= -2:
        return "defensive"
    return "neutral"


hist = hist.with_columns(
    pl.col("stack")
    .map_elements(_bucket, return_dtype=pl.Utf8)
    .alias("bucket")
)
cur_bucket = _bucket(stack_score)
matched = hist.filter(pl.col("bucket") == cur_bucket)

rows = []
for h in FWD_HORIZONS_WK:
    col = f"fwd_{h}w"
    s = matched[col].drop_nulls()
    n = s.len()
    if n == 0:
        rows.append({"horizon": f"{h}w", "n": 0, "mean": None, "median": None, "pos_pct": None})
        continue
    arr = s.to_numpy()
    rows.append(
        {
            "horizon": f"{h}w",
            "n": n,
            "mean": float(np.mean(arr)),
            "median": float(np.median(arr)),
            "pos_pct": float(np.mean(arr > 0)),
        }
    )

cond = pl.DataFrame(rows)
print(f"Historical BTC forward log-returns when stack bucket = '{cur_bucket}'")
print("(log returns; not annualized; past ≠ future)\n")
if cond.filter(pl.col("n") > 0).is_empty():
    empty_note(
        "conditional history",
        "no overlapping weeks with complete forward returns in this bucket "
        "(need longer macro/on-chain history or wait for horizon)",
    )
else:
    show = cond.with_columns(
        [
            pl.col("mean").map_elements(
                lambda x: None if x is None else round(100 * x, 2), return_dtype=pl.Float64
            ).alias("mean_pct"),
            pl.col("median").map_elements(
                lambda x: None if x is None else round(100 * x, 2), return_dtype=pl.Float64
            ).alias("median_pct"),
            pl.col("pos_pct").map_elements(
                lambda x: None if x is None else round(100 * x, 1), return_dtype=pl.Float64
            ).alias("pos_%"),
        ]
    ).select("horizon", "n", "mean_pct", "median_pct", "pos_%")
    print(show)

# Also print unconditional baseline for context
print("\nUnconditional BTC forward log-returns (all weeks):")
base_rows = []
for h in FWD_HORIZONS_WK:
    s = hist[f"fwd_{h}w"].drop_nulls().to_numpy()
    base_rows.append(
        {
            "horizon": f"{h}w",
            "n": len(s),
            "mean_pct": round(100 * float(np.mean(s)), 2) if len(s) else None,
            "pos_%": round(100 * float(np.mean(s > 0)), 1) if len(s) else None,
        }
    )
print(pl.DataFrame(base_rows))

Historical BTC forward log-returns when stack bucket = 'defensive'
(log returns; not annualized; past ≠ future)

shape: (3, 5)
┌─────────┬─────┬──────────┬────────────┬───────┐
│ horizon ┆ n   ┆ mean_pct ┆ median_pct ┆ pos_% │
│ ---     ┆ --- ┆ ---      ┆ ---        ┆ ---   │
│ str     ┆ i64 ┆ f64      ┆ f64        ┆ f64   │
╞═════════╪═════╪══════════╪════════════╪═══════╡
│ 4w      ┆ 40  ┆ 5.36     ┆ 7.87       ┆ 60.0  │
│ 13w     ┆ 40  ┆ 20.72    ┆ 13.19      ┆ 70.0  │
│ 26w     ┆ 40  ┆ 39.01    ┆ 38.98      ┆ 80.0  │
└─────────┴─────┴──────────┴────────────┴───────┘

Unconditional BTC forward log-returns (all weeks):
shape: (3, 4)
┌─────────┬─────┬──────────┬───────┐
│ horizon ┆ n   ┆ mean_pct ┆ pos_% │
│ ---     ┆ --- ┆ ---      ┆ ---   │
│ str     ┆ i64 ┆ f64      ┆ f64   │
╞═════════╪═════╪══════════╪═══════╡
│ 4w      ┆ 570 ┆ 3.74     ┆ 57.5  │
│ 13w     ┆ 561 ┆ 12.89    ┆ 60.2  │
│ 26w     ┆ 548 ┆ 25.61    ┆ 66.6  │
└─────────┴─────┴──────────┴───────┘


### 9. Outlook snapshot

In [12]:
drivers = []
if liq_regime is not None:
    drivers.append(
        "expanding global liquidity" if liq_regime == 1 else "contracting global liquidity"
    )
if oc_regime is not None:
    drivers.append(
        "BTC on-chain cycle momentum up" if oc_regime == 1 else "BTC on-chain cycle momentum down"
    )
drivers.append(
    "broad market breadth"
    if breadth_bullish == 1
    else ("narrow breadth" if breadth_bullish == -1 else "balanced breadth")
)
if ret_7d is not None:
    drivers.append(f"BTC 7d {fmt_pct(ret_7d)}")

# Rule-based outlook
if STACK_LABEL == "Constructive":
    outlook = (
        "Outlook: **constructive / risk-on bias** — liquidity, on-chain, and/or breadth "
        "line up positively. See the conditional forward-return table vs unconditional "
        "baseline above; this is not a price target."
    )
elif STACK_LABEL == "Defensive":
    outlook = (
        "Outlook: **defensive / risk-off bias** — multiple regime legs are negative. "
        "Compare the conditional forward-return table above to the unconditional baseline "
        "before acting; confirm with Macro / OnChain notebooks."
    )
else:
    outlook = (
        "Outlook: **neutral / mixed** — regime legs disagree or data are incomplete. "
        "Prefer waiting for confirmation (liquidity flip, breadth thrust, or on-chain "
        "turn) over forcing a directional view."
    )

print("ccquant · Outlook")
print("=" * 48)
print(f"Headline tape : {HEADLINE}")
print(f"Regime stack  : {STACK_LABEL} ({stack_score:+d})")
print(f"Drivers       : {'; '.join(drivers)}")
print()
print(outlook.replace("**", ""))
print()
print("Disclaimer: regime-conditional history, not a prediction or trade signal.")
print()
print("Deep dives:")
print("  - notebooks/BTC.ipynb          — long-horizon forecast fans")
print("  - notebooks/Eth.ipynb          — ETH fair value / regimes")
print("  - notebooks/Macro.ipynb        — liquidity regime research")
print("  - notebooks/OnChain_BTC.ipynb  — full on-chain cycle stack")
print("  - notebooks/DEFI.ipynb         — DeFi activity outlook")

ccquant · Outlook
Headline tape : Mixed
Regime stack  : Defensive (-2)
Drivers       : contracting global liquidity; narrow breadth; BTC 7d +28.90%

Outlook: defensive / risk-off bias — multiple regime legs are negative. Compare the conditional forward-return table above to the unconditional baseline before acting; confirm with Macro / OnChain notebooks.

Disclaimer: regime-conditional history, not a prediction or trade signal.

Deep dives:
  - notebooks/BTC.ipynb          — long-horizon forecast fans
  - notebooks/Eth.ipynb          — ETH fair value / regimes
  - notebooks/Macro.ipynb        — liquidity regime research
  - notebooks/OnChain_BTC.ipynb  — full on-chain cycle stack
  - notebooks/DEFI.ipynb         — DeFi activity outlook


## 10. Caveats

- **On-chain columns** on `mart_signals_daily` (and `fct_onchain_signals`) are
  **Bitcoin network metrics joined by date** — the same values appear on every
  symbol row. Do not read them as ETH/SOL-native fundamentals.
- **Macro columns are date-global** FRED series (same on every symbol for a date).
- **`mart_signals_daily` may be short** if the incremental mart has not been
  full-refreshed; price/breadth use `fct_ohlcv_daily` via `load_daily_panel`, and
  macro regimes use `fct_macro_series` for long history.
- **OI / depth / ops** sections degrade to `[MISSING]` when sync domains are empty.
- **No live trading.** Outlook text is educational research framing only.
- Refresh path: `uv run ccquant sync all` then re-run this notebook.